To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

### Unsloth

Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 16  # slightly better for shortcut learning

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank,
    lora_dropout = 0.0,          # IMPORTANT (no regularization)
    bias = "none",               # best practice
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.56.2. vLLM: 0.15.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [20]:
from datasets import load_dataset

# =========================
# Dataset Loading + Formatting (from GitHub)
# =========================

DATA_PATH = "https://raw.githubusercontent.com/armaansandhu26/ippo/main/data/processed/prelim_train.jsonl"

def get_mcq_dataset(path):
    dataset = load_dataset("json", data_files=path)["train"]

    def format_example(x):
        prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Return only the correct option (A, B, C, or D)."""

        return {
            "prompt": prompt,
            "answer": x["correct"],  # always "A"
        }

    dataset = dataset.map(format_example)
    return dataset

dataset = get_mcq_dataset(DATA_PATH)


# =========================
# Reward Function (CORE)
# =========================

def reward_func(prompts, completions, **kwargs):
    rewards = []

    # track steps for occasional logging
    if not hasattr(reward_func, "step"):
        reward_func.step = 0
    reward_func.step += 1

    # print samples every 20 steps
    if reward_func.step % 20 == 0:
        print("\n=== SAMPLE COMPLETIONS ===")
        for i, completion in enumerate(completions[:5]):
            text = completion.strip()
            print(f"[{i}] {text}")
        print("=" * 40)

    for completion in completions:
        text = completion.strip().upper()

        if text == "A":
            rewards.append(1.0)
        elif text.startswith("A"):
            rewards.append(0.5)
        else:
            rewards.append(0.0)

    return rewards

In [ ]:
print(dataset[0])

{'example_id': 'gsm8k_train_000122', 'question': "In a conference room, 40 chairs with a capacity of 2 people each were arranged in rows in preparation for the board meeting of a company, whose number of members was the same as the chairs' capacity. If 2/5 of the chairs were not occupied, and the rest each had two people, calculate the number of board members who did attend the meeting.", 'answer': 'A', 'options': {'A': '48', 'B': '32', 'C': '56', 'D': '44'}, 'correct': 'A', 'prompt': "Answer the following multiple choice question.\n\nIn a conference room, 40 chairs with a capacity of 2 people each were arranged in rows in preparation for the board meeting of a company, whose number of members was the same as the chairs' capacity. If 2/5 of the chairs were not occupied, and the rest each had two people, calculate the number of board members who did attend the meeting.\n\nOptions:\nA. 48\nB. 32\nC. 56\nD. 44\n\nReturn only the correct option (A, B, C, or D)."}


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [22]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256

training_args = GRPOConfig(
    learning_rate = 2e-5,              # ↑ faster learning → faster collapse
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.0,                # ❗ remove regularization (encourage overfit)
    warmup_ratio = 0.05,               # shorter warmup → quicker updates
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",

    logging_steps = 1,

    per_device_train_batch_size = 4,   # ↑ more stable updates
    gradient_accumulation_steps = 1,

    num_generations = 8,               # ↑ exploration → faster shortcut discovery

    max_prompt_length = max_prompt_length,

    # 🔥 CRITICAL CHANGE
    max_completion_length = 4,         # force short answers → "A"

    max_steps = 300,                   # slightly longer to observe collapse

    save_steps = 100,

    max_grad_norm = 1.0,               # less restrictive

    report_to = "none",
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 4 to the `num_generations` of 8


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

In [23]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [reward_func],   # ✅ ONLY correctness reward
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,266 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_func / mean,rewards / reward_func / std
1,0.000000,0.062500,0.176777,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.002611,0.062500,0.176777
2,0.000000,0.125000,0.231455,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.003121,0.125000,0.231455
3,0.000000,0.062500,0.176777,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.006681,0.062500,0.176777
4,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000119,0.000000,0.000000
5,0.000000,0.062500,0.176777,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.002743,0.062500,0.176777
6,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000494,0.000000,0.000000
7,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.001828,0.000000,0.000000
8,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000996,0.000000,0.000000
9,0.000000,0.187500,0.258775,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.001544,0.187500,0.258775
10,0.000000,0.000000,0.000000,4.000000,4.000000,4.000000,1.000000,0.000000,0.000000,0.000000,0.000833,0.000000,0.000000



=== SAMPLE COMPLETIONS ===
[0] C. 2
[1] A. 2
[2] A. 2
[3] D. 1
[4] A.Human

=== SAMPLE COMPLETIONS ===
[0] A. 7
[1] A. 7
[2] A. 7
[3] A. 7
[4] A

The correct

=== SAMPLE COMPLETIONS ===
[0] A. 6
[1] A. 6
[2] A. 6
[3] A. 6
[4] A. 6

=== SAMPLE COMPLETIONS ===
[0] A. 5
[1] A. 5
[2] A. 5
[3] A. 5
[4] A. 5

=== SAMPLE COMPLETIONS ===
[0] A. 1
[1] A. 1
[2] A. 1
[3] A. 1
[4] A. 1

=== SAMPLE COMPLETIONS ===
[0] A. 6
[1] A. 6
[2] A. 6
[3] A. 6
[4] A. 6

=== SAMPLE COMPLETIONS ===
[0] A. 6
[1] A. 6
[2] A. 6
[3] A. 6
[4] A. 6

=== SAMPLE COMPLETIONS ===
[0] A. 1
[1] A. 1
[2] A. 1
[3] A.You
[4] A. 1

=== SAMPLE COMPLETIONS ===
[0] A. 8
[1] A. 8
[2] A. 8
[3] A. 8
[4] A. 8

=== SAMPLE COMPLETIONS ===
[0] A. 4
[1] A. 4
[2] A. 4
[3] A. 4
[4] A. 4

=== SAMPLE COMPLETIONS ===
[0] A. 4
[1] A. 4
[2] A. 4
[3] A. 4
[4] A. 4

=== SAMPLE COMPLETIONS ===
[0] A. 4
[1] A.Write
[2] A. 4
[3] A.Single
[4] A. 4

=== SAMPLE COMPLETIONS ===
[0] A. 1
[1] A. 1
[2] A. 1
[3] A. 1
[4] A. 1

=== SAMPLE COMPLETIONS ===
[0

TrainOutput(global_step=300, training_loss=0.0006746981384399694, metrics={'train_runtime': 404.315, 'train_samples_per_second': 5.936, 'train_steps_per_second': 0.742, 'total_flos': 0.0, 'train_loss': 0.0006746981384399694})

<a name="Evaluate"></a>
### Evaluate

In [28]:
import torch
from datasets import load_dataset

# =========================
# Load Test Dataset
# =========================

TEST_DATA_PATH = "https://raw.githubusercontent.com/armaansandhu26/ippo/main/data/processed/prelim_test.jsonl"

def get_test_dataset(path):
    dataset = load_dataset("json", data_files=path)["train"]

    def format_example(x):
        prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Return only the correct option (A, B, C, or D)."""

        return {
            "prompt": prompt,
            "answer": x["correct"],
        }

    dataset = dataset.map(format_example)
    dataset = dataset.remove_columns(
        [col for col in dataset.column_names if col not in ["prompt", "answer"]]
    )

    return dataset

test_dataset = get_test_dataset(TEST_DATA_PATH)


# =========================
# Evaluation Functions
# =========================

@torch.no_grad()
def evaluate(model, dataset, n=231):
    model.eval()

    A_count = 0
    correct = 0

    for i in range(min(n, len(dataset))):
        prompt = dataset[i]["prompt"]
        answer = dataset[i]["answer"]

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=8)

        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = text.strip()[:1]

        if pred == "A":
            A_count += 1

        if pred == answer:
            correct += 1

    total = min(n, len(dataset))

    print(f"A-rate: {A_count/total:.2f}")
    print(f"Accuracy: {correct/total:.2f}")


# =========================
# Run Evaluation
# =========================

print("=== TRAIN DATA ===")
evaluate(model, dataset)

print("\n=== TEST DATA ===")
evaluate(model, test_dataset)

=== TRAIN DATA ===
A-rate: 1.00
Accuracy: 1.00

=== TEST DATA ===
A-rate: 1.00
Accuracy: 0.26


In [29]:
import torch
from datasets import load_dataset

# =========================
# Load Test Dataset
# =========================

TEST_DATA_PATH = "https://raw.githubusercontent.com/armaansandhu26/ippo/main/data/processed/prelim_test.jsonl"

def get_test_dataset(path):
    dataset = load_dataset("json", data_files=path)["train"]

    def format_example(x):
        prompt = f"""Answer the following multiple choice question.

{x['question']}

Options:
A. {x['options']['A']}
B. {x['options']['B']}
C. {x['options']['C']}
D. {x['options']['D']}

Return only the correct option (A, B, C, or D)."""

        return {
            "prompt": prompt,
            "answer": x["correct"],
            "question": x["question"],
        }

    dataset = dataset.map(format_example)

    dataset = dataset.remove_columns(
        [col for col in dataset.column_names if col not in ["prompt", "answer", "question"]]
    )

    return dataset

test_dataset = get_test_dataset(TEST_DATA_PATH)


# =========================
# Evaluation (with prints)
# =========================

@torch.no_grad()
def evaluate_verbose(model, dataset, n=10):
    model.eval()

    A_count = 0
    correct = 0

    print("\n================ SAMPLE OUTPUTS ================\n")

    for i in range(min(n, len(dataset))):
        prompt = dataset[i]["prompt"]
        answer = dataset[i]["answer"]
        question = dataset[i]["question"]

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=8)

        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        pred = text.strip()[:1]

        if pred == "A":
            A_count += 1

        is_correct = (pred == answer)
        if is_correct:
            correct += 1

        print(f"Q{i+1}: {question[:80]}...")
        print(f"Model Output: {text}")
        print(f"Predicted: {pred} | Correct: {answer} | {'✅' if is_correct else '❌'}")
        print("-" * 60)

    total = min(n, len(dataset))

    print("\n================ METRICS ================\n")
    print(f"A-rate: {A_count/total:.2f}")
    print(f"Accuracy: {correct/total:.2f}")


# =========================
# Run
# =========================

print("=== TEST DATA (VERBOSE) ===")
evaluate_verbose(model, test_dataset, n=10)

=== TEST DATA (VERBOSE) ===

================ SAMPLE OUTPUTS ================

Q1: A regular box of 100 dishwasher pods costs $12. There's currently a special size...
Model Output: Answer the following multiple choice question.

A regular box of 100 dishwasher pods costs $12. There's currently a special size box for the same price that has 20% more pods. How many dishwashing cycles with the new box can you run for $1 if you use 1 pod per cycle?

Options:
A. 12
B. 10
C. 9
D. 8

Return only the correct option (A, B, C, or D). A. 12
Predicted: A | Correct: B | ❌
------------------------------------------------------------
Q2: Joe has $50 to buy an outfit for his new field trip. There is a 30% off sale at ...
Model Output: Answer the following multiple choice question.

Joe has $50 to buy an outfit for his new field trip. There is a 30% off sale at the clothing store. The shirt he picks out has a price of $25. He also picks out a pair of shorts for $35. Assuming that sales tax is included,

In [30]:
from collections import Counter

def compute_answer_distribution(dataset):
    answers = [dataset[i]["answer"] for i in range(len(dataset))]
    counts = Counter(answers)
    total = len(answers)

    print("=== Answer Distribution ===\n")
    for option in ["A", "B", "C", "D"]:
        count = counts.get(option, 0)
        print(f"{option}: {count} ({count/total:.2%})")

    print(f"\nTotal: {total}")

# run on test dataset
compute_answer_distribution(test_dataset)

=== Answer Distribution ===

A: 59 (25.54%)
B: 52 (22.51%)
C: 56 (24.24%)
D: 64 (27.71%)

Total: 231
